# Phase 4.19 — Production Retrieval Pipeline

Orchestrates RRF over-retrieval, structured hard filtering, and final top-k (unit-style demo; no live index).

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "src" / "productiq").is_dir():
    ROOT = ROOT.parents[1]
sys.path.insert(0, str(ROOT / "src"))

from productiq.representation.filtering import build_filtering_representation_from_canonical
from productiq.representation.query_contract import QueryFilterConstraints, build_query_representation
from productiq.retrieval.catalog_candidate_filter import InMemoryCatalogCandidateFilter
from productiq.retrieval.contracts import RetrievalRequest, RetrievalResponse, RetrievalResponseMetadata
from productiq.retrieval.production_config import ProductionRetrievalConfig
from productiq.retrieval.production_pipeline import create_production_retrieval_pipeline
from productiq.retrieval.rrf_fusion import fused_candidate_from_rrf
from tests.database.catalog_fixtures import make_product_record

In [ ]:
class DemoRRF:
    def retrieve(self, request: RetrievalRequest) -> RetrievalResponse:
        ranking = ("P1", "P2", "P3", "P4", "P5", "P6")
        candidates = tuple(
            fused_candidate_from_rrf(
                pid,
                fusion_score=1.0 - i * 0.01,
                bm25_rank=i + 1,
                vector_rank=i + 1,
                bm25_score=1.0,
                vector_score=0.5,
            )
            for i, pid in enumerate(ranking)
        )[: request.top_k]
        return RetrievalResponse(
            candidates=candidates,
            metadata=RetrievalResponseMetadata(
                requested_top_k=request.top_k,
                fused_candidate_count=len(candidates),
                rrf_rank_constant=60,
            ),
        )

filtering_map = {}
for pid, brand in [("P1", "adidas"), ("P2", "nike"), ("P3", "adidas"), ("P4", "nike"), ("P5", "puma"), ("P6", "nike")]:
    record = make_product_record(product_id=pid, brand_normalized=brand)
    filtering_map[pid] = build_filtering_representation_from_canonical(record)

pipeline = create_production_retrieval_pipeline(
    rrf_retriever=DemoRRF(),
    catalog_filter=InMemoryCatalogCandidateFilter(filtering_map),
    config=ProductionRetrievalConfig(candidate_pool_top_k=10),
)
query = build_query_representation(
    "running shoes",
    constraints=QueryFilterConstraints(brand_normalized="nike"),
)
response = pipeline.retrieve(RetrievalRequest(query=query, top_k=3))
response.model_dump()